In [1]:
# 05_recourse_dice.ipynb
# Catastrophic-health-expenditure (CHE) risk model and algorithmic recourse.
# Shows that an observational predictive model attaches the WRONG sign to private
# insurance (adverse selection) relative to the design-based (quasi-experimental)
# Callaway-Sant'Anna estimate, and that naive DiCE recourse therefore avoids the
# design-based effective lever (acquiring
# insurance) and proposes perverse changes (lowering income / switching to medical aid).
# Robustness over DiCE methods and over proximity/diversity weights. Grayscale figures,
# no titles, 600-dpi PNG and PDF. Tables go to ../results/tables/.

import warnings, os, itertools
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score
from differences import ATTgt
import dice_ml

PROC, FIG, TAB = "../data/processed", "../results/figures", "../results/tables"
os.makedirs(FIG, exist_ok=True); os.makedirs(TAB, exist_ok=True)
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size": 11, "axes.edgecolor": "0.3", "savefig.dpi": 600,
                     "figure.dpi": 110, "grid.color": "0.9"})

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(f"{FIG}/{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)

p = pd.read_parquet(f"{PROC}/khp_panel.parquet")
num = ["age", "hh_size", "log_hhinc"]
cat = ["female", "chronic", "htn", "dm", "disabled", "HEALTH_INS", "ECO1", "EDU", "REGION1", "hh_phi_any"]
tgt = "che25"
df = p[num + cat + [tgt, "PIDWON"]].dropna().copy(); df[tgt] = df[tgt].astype(int)
for c in cat:
    df[c] = df[c].astype(int).astype(str)      # treat as categorical strings

# ---- person-level train/test split and CHE risk model ----
gss = GroupShuffleSplit(1, test_size=0.25, random_state=42)
tr, te = next(gss.split(df, groups=df["PIDWON"]))
train, test = df.iloc[tr].copy(), df.iloc[te].copy()
pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat)], remainder="passthrough")
clf = Pipeline([("pre", pre), ("gb", HistGradientBoostingClassifier(
    max_iter=350, learning_rate=0.06, max_depth=4, l2_regularization=1.0, random_state=0))])
clf.fit(train[num + cat], train[tgt])
pte = clf.predict_proba(test[num + cat])[:, 1]
perf = pd.DataFrame([{"AUC": roc_auc_score(test[tgt], pte),
                      "PR_AUC": average_precision_score(test[tgt], pte), "base_rate": test[tgt].mean()}])
perf.to_csv(f"{TAB}/t05_model_performance.csv", index=False)

# ---- associational insurance effect (flip PHI 0->1) vs causal CS effect ----
A = test.copy(); A["hh_phi_any"] = "0"; B = test.copy(); B["hh_phi_any"] = "1"
assoc = (clf.predict_proba(B[num + cat])[:, 1] - clf.predict_proba(A[num + cat])[:, 1]).mean()
d2 = p[["PIDWON", "year", "cohort", "che25", "base_phi"]].dropna(subset=["che25"]).copy()
d2["cohort"] = d2["cohort"].replace(0, np.nan); d2 = d2.set_index(["PIDWON", "year"])
att = ATTgt(data=d2, cohort_column="cohort", base_period="universal")
att.fit(formula="che25", est_method="dr", control_group="not_yet_treated", sample_split_column="base_phi", progress_bar=False)
s = att.aggregate("simple"); s.columns = [str(c) for c in s.columns]
causal = float(s.iloc[1, 0]) - float(s.iloc[0, 0])     # ATT(PHI) - ATT(no PHI); negative => protective
contrast = pd.DataFrame([{"estimate": "associational_model", "effect_pp": assoc * 100},
                         {"estimate": "design_based_DiD", "effect_pp": causal * 100}])
contrast.to_csv(f"{TAB}/t05_phi_effect_contrast.csv", index=False)

# ---- DiCE recourse across methods ----
dd = dice_ml.Data(dataframe=train[num + cat + [tgt]], continuous_features=num, outcome_name=tgt)
m = dice_ml.Model(model=clf, backend="sklearn")
unins = test[test["hh_phi_any"] == "0"].copy(); unins["risk"] = clf.predict_proba(unins[num + cat])[:, 1]
hi = unins.sort_values("risk", ascending=False).head(20)[num + cat].reset_index(drop=True)
vary = ["hh_phi_any", "HEALTH_INS", "ECO1", "log_hhinc"]
method_rows = []
for meth in ["random", "genetic", "kdtree"]:
    exp = dice_ml.Dice(dd, m, method=meth)
    kw = dict(total_CFs=2, desired_class=0, features_to_vary=vary)
    if meth == "random":
        kw["random_seed"] = 1
    cf = exp.generate_counterfactuals(hi, **kw)
    lev = {k: 0 for k in vary}; nf = 0; ncf = 0
    for i in range(len(hi)):
        c = cf.cf_examples_list[i].final_cfs_df
        if c is None or len(c) == 0:
            continue
        nf += 1; o = hi.iloc[i]
        for j in range(len(c)):
            ncf += 1
            for k in vary:
                if str(c.iloc[j][k]) != str(o[k]):
                    lev[k] += 1
    method_rows.append({"method": meth, "valid": nf, "n_cf": ncf,
                        "phi_acquire": lev["hh_phi_any"], **{f"lever_{k}": lev[k] for k in vary}})
methods_tab = pd.DataFrame(method_rows); methods_tab.to_csv(f"{TAB}/t05_dice_methods.csv", index=False)

# ---- hyperparameter sensitivity (genetic: proximity x diversity) ----
exp_g = dice_ml.Dice(dd, m, method="genetic")
hi12 = hi.head(12)
grid = list(itertools.product([0.2, 0.5, 1.0, 2.0], [0.5, 1.0, 2.0]))
sens_rows = []
for pw, dw in grid:
    cf = exp_g.generate_counterfactuals(hi12, total_CFs=3, desired_class=0, features_to_vary=vary,
                                        proximity_weight=pw, diversity_weight=dw)
    ncf = phi = income_down = welfare = 0
    for i in range(len(hi12)):
        c = cf.cf_examples_list[i].final_cfs_df
        if c is None or len(c) == 0:
            continue
        o = hi12.iloc[i]
        for j in range(len(c)):
            ncf += 1; row = c.iloc[j]
            if str(row["hh_phi_any"]) != str(o["hh_phi_any"]):
                phi += 1
            if float(row["log_hhinc"]) < float(o["log_hhinc"]) - 1e-6:
                income_down += 1
            if str(row["HEALTH_INS"]) in ("5", "6") and str(o["HEALTH_INS"]) not in ("5", "6"):
                welfare += 1
    pct = (lambda x: 100 * x / ncf if ncf else 0.0)
    sens_rows.append({"proximity_weight": pw, "diversity_weight": dw,
                      "phi_recommend_pct": pct(phi), "income_down_pct": pct(income_down), "welfare_switch_pct": pct(welfare)})
sens = pd.DataFrame(sens_rows); sens.to_csv(f"{TAB}/t05_dice_sensitivity.csv", index=False)

# ---- Figure 7: (A) associational vs causal PHI effect, (B) DiCE lever usage by method ----
fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
cvals = [assoc * 100, causal * 100]
b = ax[0].bar(["Associational\n(predictive model)", "Design-based DiD\n(Callaway-Sant'Anna)"], cvals,
              color=["0.35", "0.7"], edgecolor="0.3", width=0.55)
ax[0].axhline(0, color="0.5", lw=0.8)
for bi, v in zip(b, cvals):
    ax[0].text(bi.get_x() + bi.get_width() / 2, v + (0.1 if v >= 0 else -0.3), f"{v:+.1f}pp", ha="center", fontweight="bold")
ax[0].set_ylabel("Effect of holding PHI on P(CHE 25%), pp")
methods = methods_tab["method"].tolist(); xpos = np.arange(len(methods)); wd = 0.2
shades = {"hh_phi_any": "0.1", "HEALTH_INS": "0.4", "ECO1": "0.6", "log_hhinc": "0.8"}
for j, k in enumerate(vary):
    ax[1].bar(xpos + (j - 1.5) * wd, methods_tab[f"lever_{k}"], wd, color=shades[k], edgecolor="0.3", label=k)
ax[1].set_xticks(xpos); ax[1].set_xticklabels(methods); ax[1].set_ylabel("Times used in counterfactuals")
ax[1].set_xlabel("DiCE method"); ax[1].legend(frameon=False, fontsize=8, title="lever")
save_fig(fig, "fig07_recourse_contrast")

# ---- Figure 8: sensitivity heatmap of PHI-recommendation rate (grayscale) ----
piv = sens.pivot(index="proximity_weight", columns="diversity_weight", values="phi_recommend_pct")
fig, ax = plt.subplots(figsize=(5.5, 4.3))
sns.heatmap(piv, annot=True, fmt=".0f", cmap="Greys", vmin=0, vmax=20, cbar_kws={"label": "PHI-recommendation rate (%)"},
            linewidths=0.5, linecolor="0.8", ax=ax)
ax.set_xlabel("diversity_weight"); ax.set_ylabel("proximity_weight")
save_fig(fig, "fig08_recourse_sensitivity")

print("Model: AUC={:.3f} PR-AUC={:.3f} base={:.3f}".format(perf.iloc[0]["AUC"], perf.iloc[0]["PR_AUC"], perf.iloc[0]["base_rate"]))
print("PHI effect on CHE25 -- associational: {:+.4f} | design-based DiD: {:+.4f}".format(assoc, causal))
print("DiCE methods (PHI-acquire out of n_cf):")
print(methods_tab[["method", "valid", "n_cf", "phi_acquire"]].to_string(index=False))
print("Sensitivity PHI-recommend% range: [{:.0f}, {:.0f}]".format(sens["phi_recommend_pct"].min(), sens["phi_recommend_pct"].max()))


  0%|          | 0/20 [00:00<?, ?it/s]

  5%|▌         | 1/20 [00:00<00:03,  4.88it/s]

 10%|█         | 2/20 [00:00<00:03,  4.55it/s]

 15%|█▌        | 3/20 [00:00<00:03,  4.75it/s]

 20%|██        | 4/20 [00:00<00:03,  4.77it/s]

 25%|██▌       | 5/20 [00:01<00:03,  4.85it/s]

... (225 outputs left out)

 92%|█████████▏| 11/12 [00:09<00:00,  1.10it/s]

100%|██████████| 12/12 [00:10<00:00,  1.07it/s]

100%|██████████| 12/12 [00:10<00:00,  1.13it/s]

Model: AUC=0.846 PR-AUC=0.273 base=0.058
PHI effect on CHE25 -- associational: +0.0143 | design-based DiD: -0.0678
DiCE methods (PHI-acquire out of n_cf):
 method  valid  n_cf  phi_acquire
 random     20    40            1
genetic     20    40            0
 kdtree      4     6            1
Sensitivity PHI-recommend% range: [0, 6]
